# Комфортная маршрутизация: Konstanz и Санкт-Петербург

Ноутбук проверяет модель комфорта для велосипеда: вес ребра графа = длина × коэффициент комфорта по OSM-тегам. Библиотечный код лежит в `src/cycle_routing`.

1. **Konstanz** (поездки STADTRADELN 2024): подбираем веса по реальным потокам велосипедистов и проверяем их на отложенных OD-парах.
2. **Санкт-Петербург** (личные GPX): сравниваем маршруты моделей с тем, как вы ездите на самом деле.

В каждом разделе: несколько строк сводки, итоговые таблицы и карта 2×2, где один и тот же маршрут раскрашен по разным признакам.

In [ ]:
from copy import deepcopy
import json
from pathlib import Path
import sys
import warnings

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = next(
        parent for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]
        if (parent / "pyproject.toml").exists()
    )
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
from pyproj import Geod
from shapely.geometry import Point

from cycle_routing import (
    DEFAULT_COMFORT_CONFIG,
    DEFAULT_COMFORT_PARAMETERS,
    ComfortOptimizationConfig,
    GraphConfig,
    MatchConfig,
    TurnConfig,
    add_comfort_cost,
    build_routes,
    comfort_feature_table,
    compare_route_models,
    composition_table,
    evaluate_corridor_popularity,
    evaluate_personal_routes,
    graph_cache_path,
    load_clean_personal_gpx,
    load_or_download_graph,
    load_stadtradeln,
    match_stadtradeln_to_graph,
    nodes_near_observations,
    optimize_comfort_weights,
    route_edge_association,
    route_edges_long,
    sample_od_pairs,
    snap_tracks_to_edges,
    spatial_od_split,
    track_edge_usage,
)
from cycle_routing.visualization import (
    ROAD_CLASS_COLORS,
    SURFACE_CLASS_COLORS,
    Panel,
    route_feature_grid,
)

warnings.filterwarnings("ignore", message="GPKG: unrecognized user_version.*")
pd.set_option("display.max_columns", 60)
GEOD = Geod(ellps="WGS84")
MODELS = ["shortest", "comfort_default", "comfort_optimized"]

## 0. Параметры эксперимента

Для Konstanz используется `retain_all=True`: иначе OSMnx оставляет только крупнейшую компоненту и теряется значительная часть STADTRADELN. `simplify=True` удаляет промежуточные shape-узлы, но сохраняет геометрию и OSM-теги рёбер.

Оптимизация меняет восемь наиболее интерпретируемых весов. `residential=1` и `asphalt=1` остаются якорями масштаба. Целевая функция максимизирует реальный поток STADTRADELN, штрафует объезд более 10% и низкое покрытие данными. OD-пары делятся пространственно на train / validation / test; test не участвует в подборе.

По умолчанию активен профиль `full`: 60 OD-пар и 360 вычислений целевой функции. Он рассчитан на полноценный эксперимент и может работать 10–20 минут в зависимости от компьютера. Для быстрой проверки всех ячеек можно временно выбрать профиль `quick`.

Повороты при подборе весов выключены: немецкие потоки агрегированы без направления. Для итоговых маршрутов `TURN_CONFIG.enabled=True`, поэтому обе comfort-модели получают одинаковые штрафы поворотов.

In [ ]:
NOTEBOOK_DATA = PROJECT_ROOT / "notebooks" / "data"
OSM_DIR = NOTEBOOK_DATA / "osm"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OSM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GERMANY_GRAPH_CONFIG = GraphConfig(
    network_type="bike", simplify=True, retain_all=True, truncate_by_edge=True,
)
GERMANY_RAW_GRAPH_CONFIG = GraphConfig(
    network_type="bike", simplify=False, retain_all=True, truncate_by_edge=True,
)
SPB_GRAPH_CONFIG = GraphConfig(
    network_type="bike", simplify=True, retain_all=False, truncate_by_edge=True,
)
USE_RAW_OSM = False
ACTIVE_GERMANY_CONFIG = GERMANY_RAW_GRAPH_CONFIG if USE_RAW_OSM else GERMANY_GRAPH_CONFIG
MATCH_CONFIG = MatchConfig(
    osmid_max_distance_m=20,
    geometry_max_distance_m=15,
    aggregate_trips="length_weighted_mean",
)
CURRENT_COMFORT_CONFIG = deepcopy(DEFAULT_COMFORT_CONFIG)
TURN_CONFIG = TurnConfig(
    enabled=True,
    slight_penalty_m=8,
    right_penalty_m=20,
    left_penalty_m=30,
    u_turn_penalty_m=90,
)
OPTIMIZATION_TURN_CONFIG = TurnConfig(enabled=False)
OPTIMIZATION_PROFILE = "full"  # "full" or "quick"
OPTIMIZATION_PROFILES = {
    "full": {
        "n_od": 60,
        "maxiter": 8,
        "popsize": 5,
        "validation_candidates": 24,
    },
    "quick": {
        "n_od": 24,
        "maxiter": 3,
        "popsize": 4,
        "validation_candidates": 12,
    },
}
if OPTIMIZATION_PROFILE not in OPTIMIZATION_PROFILES:
    raise ValueError(f"Unknown optimization profile: {OPTIMIZATION_PROFILE}")
PROFILE = OPTIMIZATION_PROFILES[OPTIMIZATION_PROFILE]
OPTIMIZATION_CONFIG = ComfortOptimizationConfig(
    seed=42,
    maxiter=PROFILE["maxiter"],
    popsize=PROFILE["popsize"],
    polish=False,
    free_detour_ratio=0.10,
    detour_penalty=10.0,
    coverage_reward=0.50,
    regularization=0.05,
    validation_candidates=PROFILE["validation_candidates"],
    show_progress=True,
)

SOURCE_CRS = "EPSG:4326"
SPB_CRS = "EPSG:32636"
START_ADDRESS = "Гражданский проспект, 27 к2, Санкт-Петербург, Россия"
END_ADDRESS = "Биржевая линия, 14, Санкт-Петербург, Россия"
N_GERMAN_OD = PROFILE["n_od"]
RANDOM_SEED = 42
SAMPLE_STEP_M = 25.0
MATCH_DISTANCE_M = 25.0
ROUTE_TOLERANCE_M = 35.0

MAX_OBJECTIVE_EVALUATIONS = (
    (OPTIMIZATION_CONFIG.maxiter + 1)
    * OPTIMIZATION_CONFIG.popsize
    * len(DEFAULT_COMFORT_PARAMETERS)
)
print(
    f"Профиль {OPTIMIZATION_PROFILE}: {N_GERMAN_OD} OD-пар, до {MAX_OBJECTIVE_EVALUATIONS} "
    f"оценок целевой функции; повороты в итоговых маршрутах: {TURN_CONFIG.enabled}"
)

## 1. Konstanz (поездки STADTRADELN 2024)

STADTRADELN — ежегодная кампания, в которой участники записывают свои велопоездки. Konstanz публикует агрегат: для каждого сегмента OSM-улицы число поездок по нему. Это наша «правда» о том, где люди реально ездят.

In [ ]:
STAD_URL = (
    "https://offenedaten-konstanz.de/sites/default/files/"
    "Verkehrsmengen%202.0_SR%202024_Konstanz_UTM32_je_Wochentag_gesamt.zip"
)
STAD_DIR = NOTEBOOK_DATA / "stadtradeln_konstanz"
observed = load_stadtradeln(
    STAD_DIR,
    url=STAD_URL,
    archive_name="stadtradeln_konstanz_2024_traffic_volumes.zip",
)
trips = observed["number_of_matched_trips"]
popular_threshold = float(trips.quantile(0.75))
print(
    f"STADTRADELN: {len(observed):,} сегментов на {observed['osm_way_id'].nunique():,} OSM-путях; "
    f"поездок на сегмент: медиана {trips.median():.0f}, популярный порог (p75) = {popular_threshold:.0f}"
)

### Граф и привязка поездок

Граф скачивается из текущего OSM и упрощается, поэтому сегменты датасета приходится заново находить на его рёбрах: сначала по `osm_way_id` с проверкой геометрии, иначе ближайшее ребро в пределах 15 м. «Расхождение» — расстояние от сегмента до найденного ребра. Это только проверка, что привязка не промахнулась; число поездок при переносе не меняется. Если на ребро попало несколько сегментов, поездки усредняются с весом по длине.

In [ ]:
observed_wgs84 = observed.to_crs(SOURCE_CRS)
left, bottom, right, top = observed_wgs84.total_bounds
konstanz_bbox = (left - 0.005, bottom - 0.005, right + 0.005, top + 0.005)

de_cache = graph_cache_path(OSM_DIR, "konstanz", ACTIVE_GERMANY_CONFIG)
G_de_downloaded = load_or_download_graph(
    de_cache,
    config=ACTIVE_GERMANY_CONFIG,
    bbox=konstanz_bbox,
)
G_de = ox.projection.project_graph(G_de_downloaded, to_crs=observed.crs)
G_de = add_comfort_cost(G_de, CURRENT_COMFORT_CONFIG)
G_de, de_match_metrics, de_match_audit = match_stadtradeln_to_graph(
    G_de, observed, MATCH_CONFIG
)

m = de_match_metrics
print(f"Граф Konstanz: {G_de.number_of_nodes():,} узлов, {G_de.number_of_edges():,} рёбер")
print(
    f"Привязано {m['stad_match_rate']:.1%} сегментов "
    f"({m['stad_matched_via_osmid'] / m['stad_segments']:.0%} по osm id, "
    f"{m['stad_matched_via_geometry_only'] / m['stad_segments']:.0%} только по геометрии); "
    f"расхождение p95 = {m['p95_match_distance_m']:.1f} м; "
    f"поездки есть у {m['osm_edge_match_rate']:.0%} рёбер графа"
)

### Признаки функции комфорта

Каждое ребро получает коэффициент как произведение множителей:

`comfort_factor = max(highway × surface × cycleway × footway × maxspeed × lanes × lit, 0.35)`, `comfort_cost = length_m × comfort_factor`

| признак OSM | множитель | когда ≠ 1 |
|---|---|---|
| `highway` | `highway_factor` | всегда: тип дороги (cycleway 0.55 … primary 2.1) |
| `surface` | `surface_factor` | всегда; нет тега → 1.2 |
| `has_cycleway` | `cycleway_mult` | велополоса на дороге (не highway=cycleway) |
| `footway_without_bicycle` | `footway_mult` | тротуар без разрешения для велосипеда |
| `maxspeed_kmh` | `maxspeed_mult` | скорость выше 50 км/ч |
| `lanes` | `lanes_mult` | 3 полосы и больше |
| `unlit` | `lit_mult` | `lit=no` |

`road_class` и `surface_class` в расчёте не участвуют: это укрупнённые категории для таблиц и карт. `stad_trips` — поездки STADTRADELN, привязанные к ребру. Ниже по одному ребру каждого типа.

In [ ]:
de_features = comfort_feature_table(G_de, CURRENT_COMFORT_CONFIG)
de_features.groupby("road_class", sort=False).head(1)

### Подбор весов comfort

Differential evolution подходит здесь лучше градиентного метода: при изменении веса маршрут переключается дискретно. Оптимизатор видит только train-блок. Validation и test нужны для проверки переноса, а не для улучшения показателя задним числом.

Функция качества: средний по длине `log(1 + trips)` + бонус покрытия − квадратичный штраф за объезд свыше 10% − регуляризация отклонения от исходных весов.

In [ ]:
covered_nodes = nodes_near_observations(G_de, observed, MATCH_DISTANCE_M)
all_german_pairs = sample_od_pairs(
    G_de, covered_nodes, N_GERMAN_OD, RANDOM_SEED, prefix="DE"
)
german_splits = spatial_od_split(G_de, all_german_pairs)
print("OD-пары: " + ", ".join(f"{name} {len(pairs)}" for name, pairs in german_splits.items()))

optimization_result = optimize_comfort_weights(
    G_de,
    german_splits,
    CURRENT_COMFORT_CONFIG,
    optimization_config=OPTIMIZATION_CONFIG,
    turn_config=OPTIMIZATION_TURN_CONFIG,
)
OPTIMIZED_COMFORT_CONFIG = optimization_result.optimized_config
print(f"Поиск остановлен после {optimization_result.nfev} оценок: {optimization_result.message}")

weights = optimization_result.parameters.set_index("parameter")[["current", "optimized"]]
weights["change"] = weights["optimized"] / weights["current"] - 1
display(
    weights.rename(columns={"current": "было", "optimized": "стало", "change": "изменение"})
    .style.format({"было": "{:.2f}", "стало": "{:.2f}", "изменение": "{:+.0%}"})
)
display(
    optimization_result.comparison
    .pivot(index="split", columns="model", values="mean_score")
    .loc[["train", "validation", "test"]]
    .style.format("{:.3f}")
    .set_caption("Целевая функция (больше = лучше). Честная оценка — test.")
)

### Проверка на test

Маршруты строятся с поворотами для отложенных OD-пар, которые не участвовали в подборе. Каждый маршрут режется на точки через 25 м, и каждая точка берёт поездки ближайшего сегмента STADTRADELN.

- **объезд** — насколько маршрут длиннее кратчайшего;
- **поездок × кратчайший** — во сколько раз больше поездок вдоль маршрута, чем вдоль кратчайшего (среднее геометрическое, медиана по парам);
- **популярные улицы** — доля маршрута по сегментам выше p75;
- **отличается от comfort_default** — в скольких парах оптимизация вообще поменяла маршрут.

In [ ]:
german_pairs = german_splits["test"]
german_routes = build_routes(
    G_de,
    german_pairs,
    turn_config=TURN_CONFIG,
    comfort_configs={
        "comfort_default": CURRENT_COMFORT_CONFIG,
        "comfort_optimized": OPTIMIZED_COMFORT_CONFIG,
    },
)
german_metrics = evaluate_corridor_popularity(
    german_routes,
    observed,
    sample_step_m=SAMPLE_STEP_M,
    match_distance_m=MATCH_DISTANCE_M,
    popular_threshold=popular_threshold,
)
german_comparison = compare_route_models(german_metrics)

default_edges = german_routes.query("algorithm == 'comfort_default'").set_index("od_id")["edge_route"]
german_routes["differs_from_default"] = [
    row.edge_route != default_edges[row.od_id] for row in german_routes.itertuples()
]
n_test = german_metrics["od_id"].nunique()
german_summary = pd.DataFrame({
    model: {
        "объезд, % (медиана)": comparison["detour_pct"].median() if len(comparison) else 0.0,
        "поездок × кратчайший": np.exp(comparison["popularity_gain"].median()) if len(comparison) else 1.0,
        "популярные улицы, %": 100 * german_metrics.query("algorithm == @model")["popular_edge_share"].median(),
        "популярнее кратчайшего": f"{(comparison['popularity_gain'] > 0).sum()}/{n_test}" if len(comparison) else "—",
        "отличается от comfort_default": f"{german_routes.query('algorithm == @model')['differs_from_default'].sum()}/{n_test}",
    }
    for model in MODELS
    for comparison in [german_comparison.query("model == @model")]
}).T
german_summary.style.format({
    "объезд, % (медиана)": "{:.1f}",
    "поездок × кратчайший": "{:.2f}",
    "популярные улицы, %": "{:.0f}",
})

### Насколько выбор рёбер следует популярности

Для каждой OD-пары берётся коридор: все улицы в 300 м от любого из маршрутов. Внутри коридора сравниваем, какие улицы модель выбрала, с числом поездок по ним.

- **корреляция** — взвешенная по длине корреляция «ребро выбрано (0/1)» с `log(1 + поездки)`. Больше нуля — модель выбирает популярные улицы;
- **популярные рёбра, %** — доля длины маршрута по улицам выше p75. Строка «коридор» показывает ту же долю для всех улиц, это уровень случайного выбора;
- **поездок × коридор** — среднее число поездок на выбранных улицах относительно среднего по коридору.

In [ ]:
ASSOCIATION_COLUMNS = {
    "corr_selected_vs_log_popularity": "корреляция",
    "popular_length_share": "популярные рёбра, %",
    "popularity_lift": "поездок × коридор",
}
ASSOCIATION_FORMAT = {"корреляция": "{:.2f}", "популярные рёбра, %": "{:.0%}", "поездок × коридор": "{:.2f}"}

german_association = route_edge_association(
    german_routes, de_features, "stad_trips", popular_threshold=popular_threshold
)
german_association.rename(columns=ASSOCIATION_COLUMNS).style.format(ASSOCIATION_FORMAT, na_rep="—")

### Карта примера

Берётся тестовая пара, где три модели дали больше всего разных маршрутов. Цвет обводки — модель, цвет линии — признак ребра. Карты синхронизированы: сдвиг или зум одной двигает все четыре.

In [ ]:
distinct_routes = german_routes.groupby("od_id")["edge_route"].agg(
    lambda routes: len({tuple(map(tuple, route)) for route in routes})
)
best_gain = german_comparison.groupby("od_id")["popularity_gain"].max()
german_example_od = (
    pd.DataFrame({"distinct": distinct_routes, "gain": best_gain})
    .sort_values(["distinct", "gain"], ascending=False)
    .index[0]
)
_, de_origin, de_destination = next(item for item in german_pairs if item[0] == german_example_od)
de_od = gpd.GeoSeries(
    [Point(G_de.nodes[node]["x"], G_de.nodes[node]["y"]) for node in (de_origin, de_destination)],
    crs=G_de.graph["crs"],
).to_crs(SOURCE_CRS)

germany_route_map = route_feature_grid(
    G_de,
    german_routes.query("od_id == @german_example_od"),
    de_features,
    [
        Panel("Популярность (поездки STADTRADELN)", "stad_trips", log=True),
        Panel("Тип дороги", "road_class", categories=ROAD_CLASS_COLORS),
        Panel("Покрытие", "surface_class", categories=SURFACE_CLASS_COLORS),
        Panel("Коэффициент комфорта (исходные веса)", "comfort_factor", low_label="комфортно", high_label="некомфортно"),
    ],
    (de_od.y.iloc[0], de_od.x.iloc[0]),
    (de_od.y.iloc[1], de_od.x.iloc[1]),
)
print(f"OD-пара {german_example_od}")
germany_route_map

## 2. Санкт-Петербург (личные GPX)

Здесь данных о чужих поездках нет. Роль популярности играют ваши треки: сколько из них проходит по ребру (точки трека через 25 м привязываются к ближайшему ребру в пределах 25 м).

In [ ]:
personal = load_clean_personal_gpx(PROJECT_ROOT / "cicle_gpx")
personal_spb = personal.to_crs(SPB_CRS)
track_km = personal_spb.geometry.length / 1000

start_place = ox.geocoder.geocode_to_gdf(START_ADDRESS, which_result=1).iloc[0]
if start_place["class"] != "building" or start_place["addresstype"] != "building":
    raise ValueError(
        f"OSM returned {start_place['class']}/{start_place['addresstype']} instead of building"
    )
start_latlon = (float(start_place["lat"]), float(start_place["lon"]))
end_latlon = ox.geocoder.geocode(END_ADDRESS)
print(
    f"GPX: {len(personal)} треков, длина {track_km.median():.1f} км (медиана), "
    f"{track_km.min():.1f}–{track_km.max():.1f} км; адреса старта и финиша найдены"
)

In [ ]:
bounds = personal.total_bounds
all_latlon = [start_latlon, end_latlon, (bounds[1], bounds[0]), (bounds[3], bounds[2])]
center = (
    (min(point[0] for point in all_latlon) + max(point[0] for point in all_latlon)) / 2,
    (min(point[1] for point in all_latlon) + max(point[1] for point in all_latlon)) / 2,
)
radius_m = max(
    abs(GEOD.inv(center[1], center[0], point[1], point[0])[2])
    for point in all_latlon
) + 1200

spb_cache = graph_cache_path(OSM_DIR, "saint_petersburg", SPB_GRAPH_CONFIG)
G_spb_downloaded = load_or_download_graph(
    spb_cache,
    config=SPB_GRAPH_CONFIG,
    center=center,
    dist=radius_m,
)
G_spb = ox.truncate.largest_component(G_spb_downloaded, strongly=True)
G_spb = ox.projection.project_graph(G_spb, to_crs=SPB_CRS)
G_spb = add_comfort_cost(G_spb, CURRENT_COMFORT_CONFIG)

spb_features = comfort_feature_table(G_spb, CURRENT_COMFORT_CONFIG)
personal_edges = snap_tracks_to_edges(
    G_spb, personal_spb, step_m=SAMPLE_STEP_M, max_distance_m=MATCH_DISTANCE_M
)
spb_features["gpx_tracks"] = track_edge_usage(personal_edges, spb_features)
print(
    f"Граф СПб: {G_spb.number_of_nodes():,} узлов, {G_spb.number_of_edges():,} рёбер; "
    f"ваши треки проходят по {int((spb_features['gpx_tracks'] > 0).sum()):,} рёбрам"
)

### Совпадение маршрутов с вашими треками

Для каждого трека строятся маршруты трёх моделей между его началом и концом.

- **F1** — насколько маршрут совпадает с треком: точки одной линии в пределах 35 м от другой, в обе стороны;
- **лучше кратчайшего** — у скольких треков F1 выше, чем у кратчайшего маршрута;
- **состав маршрута** — доля длины по типам дорог, включая ваши треки. Отсюда видно, почему модели расходятся с реальностью.

In [ ]:
personal_metrics = evaluate_personal_routes(
    G_spb,
    personal_spb,
    step_m=SAMPLE_STEP_M,
    tolerance_m=ROUTE_TOLERANCE_M,
    turn_config=TURN_CONFIG,
    comfort_configs={
        "comfort_default": CURRENT_COMFORT_CONFIG,
        "comfort_optimized": OPTIMIZED_COMFORT_CONFIG,
    },
)
f1 = personal_metrics.pivot(index="activity_id", columns="algorithm", values="f1")
detour = personal_metrics.pivot(index="activity_id", columns="algorithm", values="detour_pct")
spb_summary = pd.DataFrame({
    "F1 (медиана)": f1.median(),
    "лучше кратчайшего": {model: f"{(f1[model] > f1['shortest']).sum()}/{len(f1)}" for model in MODELS},
    "объезд, % (медиана)": detour.median(),
}).loc[MODELS]
display(spb_summary.style.format({"F1 (медиана)": "{:.2f}", "объезд, % (медиана)": "{:.1f}"}))

personal_routes = personal_metrics.rename(columns={"activity_id": "od_id"})
spb_composition = composition_table(
    pd.concat([
        personal_edges.assign(group="ваши треки"),
        route_edges_long(personal_routes, spb_features),
    ]),
    spb_features,
    "road_class",
).loc[["ваши треки", *MODELS]]
display(spb_composition.style.format("{:.0f}").set_caption("Состав маршрутов, % длины"))

spb_association = route_edge_association(
    personal_routes, spb_features, "gpx_tracks", popular_threshold=2
)
display(
    spb_association.rename(columns={**ASSOCIATION_COLUMNS, "popularity_lift": "ваших треков × коридор"})
    .style.format({**ASSOCIATION_FORMAT, "ваших треков × коридор": "{:.2f}"}, na_rep="—")
    .set_caption("Связь выбора рёбер с вашими треками (популярное ребро = минимум 2 трека)")
)

### Карта маршрута по адресам

In [ ]:
address_points = gpd.GeoSeries(
    [Point(start_latlon[1], start_latlon[0]), Point(end_latlon[1], end_latlon[0])],
    crs=SOURCE_CRS,
).to_crs(SPB_CRS)
address_nodes = ox.distance.nearest_nodes(
    G_spb,
    X=address_points.x.to_numpy(),
    Y=address_points.y.to_numpy(),
)
address_routes = build_routes(
    G_spb,
    [("SPB-address", int(address_nodes[0]), int(address_nodes[1]))],
    turn_config=TURN_CONFIG,
    comfort_configs={
        "comfort_default": CURRENT_COMFORT_CONFIG,
        "comfort_optimized": OPTIMIZED_COMFORT_CONFIG,
    },
)
spb_route_map = route_feature_grid(
    G_spb,
    address_routes,
    spb_features,
    [
        Panel(f"Ваши треки по ребру (из {len(personal)})", "gpx_tracks", low_label="0", high_label="много"),
        Panel("Тип дороги", "road_class", categories=ROAD_CLASS_COLORS),
        Panel("Покрытие", "surface_class", categories=SURFACE_CLASS_COLORS),
        Panel("Коэффициент комфорта (исходные веса)", "comfort_factor", low_label="комфортно", high_label="некомфортно"),
    ],
    start_latlon,
    end_latlon,
)
spb_route_map

## 3. Сохранение результатов

In [ ]:
pd.Series(de_match_metrics).to_csv(OUTPUT_DIR / "germany_match_quality.csv")
de_match_audit.drop(columns="geometry").to_csv(OUTPUT_DIR / "germany_match_audit.csv", index=False)
optimization_metadata = {
    "profile": OPTIMIZATION_PROFILE,
    "od_pairs": N_GERMAN_OD,
    "split_sizes": {name: len(pairs) for name, pairs in german_splits.items()},
    "parameter_count": len(DEFAULT_COMFORT_PARAMETERS),
    "max_objective_evaluations": MAX_OBJECTIVE_EVALUATIONS,
    "actual_objective_evaluations": optimization_result.nfev,
    "scipy_converged": optimization_result.success,
    "scipy_message": optimization_result.message,
    "turns_enabled_during_fit": OPTIMIZATION_TURN_CONFIG.enabled,
    "turns_enabled_during_final_comparison": TURN_CONFIG.enabled,
}
(OUTPUT_DIR / "optimization_run_metadata.json").write_text(
    json.dumps(optimization_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
optimization_result.parameters.to_csv(OUTPUT_DIR / "comfort_weight_comparison.csv", index=False)
optimization_result.comparison.to_csv(OUTPUT_DIR / "comfort_optimization_splits.csv", index=False)
optimization_result.history.to_csv(OUTPUT_DIR / "comfort_optimization_history.csv", index=False)
(OUTPUT_DIR / "optimized_comfort_config.json").write_text(
    json.dumps(OPTIMIZED_COMFORT_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8"
)
german_metrics.to_csv(OUTPUT_DIR / "germany_corridor_metrics.csv", index=False)
german_comparison.to_csv(OUTPUT_DIR / "germany_corridor_experiment.csv", index=False)
german_summary.to_csv(OUTPUT_DIR / "germany_test_summary.csv")
german_association.to_csv(OUTPUT_DIR / "germany_edge_popularity_association.csv")
personal_metrics.drop(columns="edge_route").to_csv(OUTPUT_DIR / "spb_personal_route_experiment.csv", index=False)
spb_summary.to_csv(OUTPUT_DIR / "spb_personal_summary.csv")
spb_composition.to_csv(OUTPUT_DIR / "spb_route_composition.csv")
spb_association.to_csv(OUTPUT_DIR / "spb_edge_track_association.csv")
address_routes[[
    "od_id", "algorithm", "route_length_m", "objective_cost", "turn_penalty_total_m",
]].to_csv(OUTPUT_DIR / "spb_address_route_candidates.csv", index=False)
germany_route_map.save(str(OUTPUT_DIR / "germany_route_features.html"))
spb_route_map.save(str(OUTPUT_DIR / "spb_route_features.html"))
print(f"Сохранено в {OUTPUT_DIR}")